In [36]:
import pandas as pd
import pyarrow as pa

from deltalake import DeltaTable
from deltalake import write_deltalake

In [37]:
BRONZE_PATH = "data/bronze/patient_readings"
SILVER_PATH = "data/silver/patient_readings"
GOLD_PATH = "data/gold/ward_summary"

## Bronze

Stores validated raw patient records exactly as received from the ingestion layer.

In [60]:
bronze_df = pd.DataFrame(
    [
        {
            "patient_id": "P001",
            "ward": "ICU",
            "heart_rate": 82,
            "oxygen_level": 98,
            "event_time": "2026-07-22 10:00:00"
        },
        {
            "patient_id": "P001",
            "ward": "ICU",
            "heart_rate": 84,
            "oxygen_level": 97,
            "event_time": "2026-07-22 10:05:00"
        },
        {
            "patient_id": "P002",
            "ward": "ICU",
            "heart_rate": 90,
            "oxygen_level": 95,
            "event_time": "2026-07-22 10:01:00"
        },
        {
            "patient_id": "P003",
            "ward": "Ward-A",
            "heart_rate": 75,
            "oxygen_level": 99,
            "event_time": "2026-07-22 10:02:00"
        }
    ]
)


write_deltalake(
    BRONZE_PATH,
    bronze_df,
    mode="overwrite"
)

print("Bronze Created")

dt = DeltaTable(BRONZE_PATH)

dt.to_pandas()

Bronze Created


,patient_id,ward,heart_rate,oxygen_level,event_time
0,P001,ICU,82,98,2026-07-22 10:00:00
1,P001,ICU,84,97,2026-07-22 10:05:00
2,P002,ICU,90,95,2026-07-22 10:01:00
3,P003,Ward-A,75,99,2026-07-22 10:02:00


## silver

Stores cleaned and deduplicated patient records,
keeping the latest record per patient.

In [40]:
bronze_df = DeltaTable(
    BRONZE_PATH
).to_pandas()

bronze_df["event_time"] = pd.to_datetime(
    bronze_df["event_time"]
)

bronze_df

,patient_id,ward,heart_rate,oxygen_level,event_time
0,P001,ICU,82,98,2026-07-22 10:00:00
1,P001,ICU,84,97,2026-07-22 10:05:00
2,P002,ICU,90,95,2026-07-22 10:01:00
3,P003,Ward-A,75,99,2026-07-22 10:02:00


In [45]:
silver_df = (
    bronze_df
    .sort_values("event_time")
    .drop_duplicates(
        subset=["patient_id"],
        keep="last"
    )
    .copy()
)

silver_df["event_time"] = silver_df["event_time"].astype(str)

silver_df

,patient_id,ward,heart_rate,oxygen_level,event_time
2,P002,ICU,90,95,2026-07-22 10:01:00
3,P003,Ward-A,75,99,2026-07-22 10:02:00
1,P001,ICU,84,97,2026-07-22 10:05:00


In [46]:
write_deltalake(
    SILVER_PATH,
    silver_df,
    mode="overwrite"
)

print("Silver Created")

Silver Created


In [47]:
DeltaTable(
    SILVER_PATH
).to_pandas()

,patient_id,ward,heart_rate,oxygen_level,event_time,__index_level_0__
0,P002,ICU,90,95,2026-07-22 10:01:00,2
1,P003,Ward-A,75,99,2026-07-22 10:02:00,3
2,P001,ICU,84,97,2026-07-22 10:05:00,1


## MERGE

Demonstrates Delta Lake upsert functionality using
patient_id as the business key.

In [48]:
updates_df = pd.DataFrame(
    [
        {
            "patient_id": "P002",
            "ward": "ICU",
            "heart_rate": 110,
            "oxygen_level": 92,
            "event_time": "2026-07-22 10:15:00"
        },
        {
            "patient_id": "P004",
            "ward": "Ward-B",
            "heart_rate": 78,
            "oxygen_level": 99,
            "event_time": "2026-07-22 10:20:00"
        }
    ]
)

updates_df

,patient_id,ward,heart_rate,oxygen_level,event_time
0,P002,ICU,110,92,2026-07-22 10:15:00
1,P004,Ward-B,78,99,2026-07-22 10:20:00


In [51]:
from deltalake import DeltaTable

silver_table = DeltaTable(SILVER_PATH)

In [52]:
silver_table.merge(
    source=pa.Table.from_pandas(updates_df),
    predicate="target.patient_id = source.patient_id",
    source_alias="source",
    target_alias="target"
).when_matched_update_all(
).when_not_matched_insert_all(
).execute()

{'num_source_rows': 2,
 'num_target_rows_inserted': 1,
 'num_target_rows_updated': 1,
 'num_target_rows_deleted': 0,
 'num_target_rows_copied': 2,
 'num_output_rows': 4,
 'num_target_files_scanned': 1,
 'num_target_files_skipped_during_scan': 0,
 'num_target_files_added': 1,
 'num_target_files_removed': 1,
 'execution_time_ms': 20,
 'scan_time_ms': 4,
 'rewrite_time_ms': 0}

In [53]:
DeltaTable(
    SILVER_PATH
).to_pandas()

,patient_id,ward,heart_rate,oxygen_level,event_time,__index_level_0__
0,P004,Ward-B,78,99,2026-07-22 10:20:00,NaN
1,P002,ICU,110,92,2026-07-22 10:15:00,2.0
2,P003,Ward-A,75,99,2026-07-22 10:02:00,3.0
3,P001,ICU,84,97,2026-07-22 10:05:00,1.0


## gold

Stores aggregated ward-level metrics including average heart rate, average oxygen level, and patient counts.

In [54]:
silver_df = DeltaTable(
    SILVER_PATH
).to_pandas()

silver_df

,patient_id,ward,heart_rate,oxygen_level,event_time,__index_level_0__
0,P004,Ward-B,78,99,2026-07-22 10:20:00,NaN
1,P002,ICU,110,92,2026-07-22 10:15:00,2.0
2,P003,Ward-A,75,99,2026-07-22 10:02:00,3.0
3,P001,ICU,84,97,2026-07-22 10:05:00,1.0


In [55]:
gold_df = (
    silver_df
    .groupby("ward")
    .agg(
        avg_heart_rate=("heart_rate", "mean"),
        avg_oxygen_level=("oxygen_level", "mean"),
        patient_count=("patient_id", "count")
    )
    .reset_index()
)

gold_df

,ward,avg_heart_rate,avg_oxygen_level,patient_count
0,ICU,97.0,94.5,2
1,Ward-A,75.0,99.0,1
2,Ward-B,78.0,99.0,1


In [56]:
write_deltalake(
    GOLD_PATH,
    gold_df,
    mode="overwrite"
)

print("Gold Created")

Gold Created


In [57]:
DeltaTable(
    GOLD_PATH
).to_pandas()

,ward,avg_heart_rate,avg_oxygen_level,patient_count
0,ICU,97.0,94.5,2
1,Ward-A,75.0,99.0,1
2,Ward-B,78.0,99.0,1


## Schema Enforcement

Demonstrates Delta Lake rejecting writes that do not
match the existing table schema.

In [58]:
wrong_schema_df = pd.DataFrame(
    [
        {
            "patient_id": "P999",
            "ward": "ICU",
            "temperature": 39
        }
    ]
)

wrong_schema_df

,patient_id,ward,temperature
0,P999,ICU,39


In [59]:
try:
    write_deltalake(
        SILVER_PATH,
        wrong_schema_df,
        mode="append"
    )

    print("Write succeeded")

except Exception as e:
    print("Schema Enforcement Worked")
    print(type(e).__name__)
    print(e)

Schema Enforcement Worked
SchemaMismatchError
Cannot cast schema, number of fields does not match: 3 vs 6
